## Importar librerías y definir rutas

In [1]:
import re
import os
import pandas as pd
from pathlib import Path

# Rutas de entrada y salida
INPUT_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/txt_bruto")
OUTPUT_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/txt_limpio")
METADATA_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/metadatos")

# Crear carpetas de salida
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(METADATA_FOLDER, exist_ok=True)

# Listar archivos de texto bruto
txt_files = sorted(INPUT_FOLDER.glob("*.txt"))
print(f"Archivos a limpiar: {len(txt_files)}")
for f in txt_files:
    print(f"  - {f.name}")

Archivos a limpiar: 49
  - DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025.txt
  - ESTATUTO 2024. 04-09-2024.txt
  - Guía para la organización y orientación del legajo para la docencia ordinaria.txt
  - MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021.txt
  - Modelo de índice del contenido - Legajo.txt
  - Politica Institucional de Inclusión y diversidad cultural v1.txt
  - Politica Institucional de trabajo digno y protección de la persona v.1.txt
  - Politica-ambiental.txt
  - REGLAMENTO ADMISION 2025.v7.txt
  - REGLAMENTO BECAS 2021 ACTUALIZADO.txt
  - REGLAMENTO CODIGO ETICA INVESTIGACION 2021.txt
  - REGLAMENTO DE ESTUDIOS POSGRADO 2025.txt
  - REGLAMENTO DE ESTUDIOS V5_2025.txt
  - REGLAMENTO DEFENSORIA UNIVERSITARIA 2025 v4.txt
  - REGLAMENTO DOCENCIA ORDINARIA v3.5.txt
  - REGLAMENTO ESTUDIANTE UNIONISTA V3.txt
  - REGLAMENTO GENERAL UPeU 2023.txt
  - REGLAMENTO GRADOS Y TITULOS.v8 2025.txt
  - REGLAMENTO IDENTIDAD VISUAL Y REDES SOCIALES.txt
  - REGLAMENTO

## Función de limpieza del texto

In [2]:
import re
from collections import Counter

def limpiar_texto(texto, nombre_archivo=""):
    """
    Limpia texto extraído de PDFs UPeU.

    Issues corregidos:
    - Whitelist de simbolos que incluye °, º, $, % (Issue 2.1)
    - Reconstruye parrafos sin fusionar items numerados (Issue 2.2)
    - Elimina marcadores [Pagina N] y --- Pagina N --- (Issue 2.3)
    """
    lineas = texto.splitlines()

    # 0. Detectar y eliminar encabezados/pies repetidos (>=3 veces)
    contador = Counter(l.strip() for l in lineas if 5 < len(l.strip()) < 120)
    headers_repetidos = {l for l, c in contador.items() if c >= 3}
    lineas = [l for l in lineas if l.strip() not in headers_repetidos]
    texto = "\n".join(lineas)

    # 1. Eliminar marcadores del notebook 1
    texto = re.sub(r'\[P[aá]gina\s+\d+[^\]]*\]', '', texto, flags=re.IGNORECASE)
    texto = re.sub(r'---\s*P[aá]g(?:ina)?\.?\s*\d+(?:\s*\(OCR\))?\s*---', '', texto, flags=re.IGNORECASE)
    texto = re.sub(r'\[TABLA[^\]]*\]', '', texto)

    # 2. Eliminar lineas con solo numeros (folios)
    texto = re.sub(r'^\s*\d+\s*$', '', texto, flags=re.MULTILINE)

    # 3. Eliminar headers/footers repetitivos
    texto = re.sub(r'^Universidad Peruana Uni[oó]n.*$', '', texto, flags=re.MULTILINE | re.IGNORECASE)
    texto = re.sub(r'^UPeU\s*$', '', texto, flags=re.MULTILINE)
    texto = re.sub(r'www\.upeu\.edu\.pe', '', texto, flags=re.IGNORECASE)

    # 4. Reducir espacios multiples
    texto = re.sub(r'[ \t]+', ' ', texto)

    # >>> Issue 2.1: whitelist de simbolos que incluye $, %, °, º, comillas <<<
    texto = re.sub(
        r"[^\w\sáéíóúüñÁÉÍÓÚÜÑ.,;:()\-$/%°º«»“”'\u2018\u2019]",
        '', texto, flags=re.UNICODE
    )

    # 5. Reconstruir parrafos sin fusionar items numerados ni articulos
    lineas = texto.split('\n')
    parrafos = []
    buffer = []
    for linea in lineas:
        linea = linea.strip()
        if not linea:
            if buffer:
                parrafos.append(' '.join(buffer))
                buffer = []
            parrafos.append('')
            continue

        es_inicio_item = bool(re.match(r'^\d+[\.)]\s', linea)) or bool(re.match(r'^[a-z][\.)]\s', linea))
        es_inicio_bloque = bool(re.match(r'^(Art[íi]culo|Cap[íi]tulo|Secci[óo]n|T[íi]tulo)\s', linea, re.IGNORECASE))

        if es_inicio_item or es_inicio_bloque:
            if buffer:
                parrafos.append(' '.join(buffer))
                buffer = []
            buffer.append(linea)
        elif re.search(r'[.?:!;]$', linea):
            buffer.append(linea)
            parrafos.append(' '.join(buffer))
            buffer = []
        else:
            buffer.append(linea)
    if buffer:
        parrafos.append(' '.join(buffer))

    texto = '\n\n'.join(p for p in parrafos if p is not None)
    texto = re.sub(r'\n\s*\n\s*\n+', '\n\n', texto)

    return texto.strip()


## Aplicar limpieza y guardar textos limpios

In [3]:
metadata_rows = []

for txt_path in txt_files:
    print(f"Limpiando: {txt_path.name}")
    with open(txt_path, "r", encoding="utf-8") as f:
        texto_crudo = f.read()
    
    # Limpiar
    texto_limpio = limpiar_texto(texto_crudo, txt_path.stem)
    
    # Guardar versión limpia
    output_path = OUTPUT_FOLDER / txt_path.name
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(texto_limpio)
    
    # Registrar metadatos básicos
    metadata_rows.append({
        "documento": txt_path.stem,
        "caracteres_crudo": len(texto_crudo),
        "caracteres_limpio": len(texto_limpio),
        "reduccion_pct": round(100 * (1 - len(texto_limpio)/max(len(texto_crudo),1)), 1)
    })
    print(f"  -> {len(texto_limpio)} caracteres (reducción {metadata_rows[-1]['reduccion_pct']}%)")

# Crear DataFrame y guardar
df_meta = pd.DataFrame(metadata_rows)
df_meta.to_csv(METADATA_FOLDER / "metadatos_limpieza.csv", index=False, encoding="utf-8")
print("\nMetadatos guardados en metadatos_limpieza.csv")
df_meta.head()

Limpiando: DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025.txt
  -> 26515 caracteres (reducción 6.6%)
Limpiando: ESTATUTO 2024. 04-09-2024.txt
  -> 112266 caracteres (reducción 3.3%)
Limpiando: Guía para la organización y orientación del legajo para la docencia ordinaria.txt
  -> 37120 caracteres (reducción 11.2%)
Limpiando: MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021.txt
  -> 17628 caracteres (reducción 2.3%)
Limpiando: Modelo de índice del contenido - Legajo.txt
  -> 4714 caracteres (reducción 9.6%)
Limpiando: Politica Institucional de Inclusión y diversidad cultural v1.txt
  -> 3462 caracteres (reducción 2.2%)
Limpiando: Politica Institucional de trabajo digno y protección de la persona v.1.txt
  -> 3477 caracteres (reducción 2.6%)
Limpiando: Politica-ambiental.txt
  -> 2411 caracteres (reducción 0.8%)
Limpiando: REGLAMENTO ADMISION 2025.v7.txt
  -> 102473 caracteres (reducción 11.3%)
Limpiando: REGLAMENTO BECAS 2021 ACTUALIZADO.txt
  -> 68681 caracteres (

,documento,caracteres_crudo,caracteres_limpio,reduccion_pct
0,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,28380,26515,6.6
1,ESTATUTO 2024. 04-09-2024,116069,112266,3.3
2,Guía para la organización y orientación del le...,41791,37120,11.2
3,MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILL...,18038,17628,2.3
4,Modelo de índice del contenido - Legajo,5215,4714,9.6


## Verificación manual (muestra de un archivo)

In [4]:
# Elige un archivo para revisar una muestra
import random
muestra = random.choice(list(OUTPUT_FOLDER.glob("*.txt")))
with open(muestra, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"Muestra de {muestra.name}:\n")
print(contenido[:1000])  # primeros 1000 caracteres

Muestra de REGLAMENTO DEFENSORIA UNIVERSITARIA 2025 v4.txt:

REGLAMENTO DE LA DEFENSORÍA UNIVERSITARIA

(Resolución N° 3172-2025/UPeU-CU, 29 de setiembre de 2025) (Resolución N° 2757-2025/UPeU-CU, 08 de agosto de 2025) (Resolución N° 0346-2024/UPeU-CU, 19 de enero de 2024) (Resolución N° 0035-2015/UPeU-CU, 08 de enero de 2015)

LIMA- PERÚ

Z UPeU

a UNIVERSIDAD PERUANA UNIÓN

AjUPeU

“Año de la recuperación y consolidación de la economía peruana” Ñaña, Lima, 29 de setiembre de 2025

Se ha expedido la RESOLUCIÓN N 3172-2025/UPeU-CU; que sigue:

29 de setiembre de 2025;

VISTA el acta del Consejo Universitario, del 29 de setiembre de 2025;

CONSIDERANDO:

Que la Universidad Peruana Unión (UPeU), tiene autonomía académica, de gobierno, económica, administrativa y normativa, dentro del ámbito establecido por la Ley Universitaria N” 30220, el Estatuto y el Reglamento General de la Universidad;

Que la UPeU tiene la potestad de modificar y actualizar sus reglamentos que regulan el quehacer y

In [5]:
import re

archivo = OUTPUT_FOLDER / "REGLAMENTO DE ESTUDIOS V5_2025.txt"

with open(archivo, "r", encoding="utf-8") as f:
    contenido = f.read()

match = re.search(
    r"(Artículo\s+1[º°o]?.*?)(?=Artículo\s+2|\Z)",
    contenido,
    re.DOTALL | re.IGNORECASE
)

if match:
    print(match.group(1))
else:
    print("No se encontró el Artículo 1")

Artículo 1º UPeU y sus unidades académicas. La Universidad Peruana Unión (UPeU) gestiona y desarrolla sus actividades de formación profesional y especialización, en sus modalidades de presencial, semipresencial y a distancia, a través de sus programas de estudios de sus Facultades y escuelas profesionales (EP), y los centros universitarios, las unidades de apoyo y servicio académico creados y facultados para tal objeto como el Centro de Idiomas (CI) y el Conservatorio de Música (CM).


